# Lesson 06-1：SQLite 新手練習

這份 notebook 使用 `data/raw/course.db`，用 5 段小程式練習 SQLite 的基本操作。

學習目標：
- 用 `sqlite3` 連線到 `.db` 檔案
- 使用 `SELECT`、`WHERE`、`ORDER BY`、`LIMIT` 查詢資料
- 使用 `GROUP BY` 做分組統計
- 建立練習用暫存表並新增資料

本 notebook 的新增資料練習只會寫入記憶體中的暫存資料庫，不會修改原始 `course.db`。

## 大綱

1. 連線資料庫並查看有哪些資料表
2. 基本查詢：選欄位與限制筆數
3. 條件查詢：`WHERE`、日期、文字條件
4. 分組彙總：`GROUP BY`、`COUNT`、`SUM`、`AVG`
5. 新增資料：建立練習表、`INSERT`、再查詢確認

## 0. 準備：匯入套件與設定資料庫路徑

先載入 `sqlite3` 和 `pandas`。`sqlite3` 負責連線資料庫，`pandas.read_sql_query()` 可以把 SQL 查詢結果變成表格。

In [1]:
from pathlib import Path
import sqlite3
import pandas as pd

DB_PATH = Path("data/raw/course.db")
DB_PATH.exists(), DB_PATH

(True, WindowsPath('data/raw/course.db'))

## 1. 連線資料庫並查看資料表

`sqlite_master` 是 SQLite 內建的目錄表，可以用來查詢這個資料庫有哪些 table。

In [2]:
with sqlite3.connect(DB_PATH) as conn:
    tables = pd.read_sql_query(
        """
        SELECT name AS table_name
        FROM sqlite_master
        WHERE type = 'table'
        ORDER BY name;
        """,
        conn,
    )

tables

,table_name
0,ab_assignments
1,customers
2,events
3,order_items
4,orders
5,products
6,sessions


也可以先看幾筆資料，理解欄位長什麼樣子。這裡以 `orders` 訂單表為例。

In [6]:
with sqlite3.connect(DB_PATH) as conn:
    orders_preview = pd.read_sql_query(
        """
        SELECT order_id, customer_id, order_date, status, payment_type
        FROM orders
        WHERE status = 'cancelled'
        LIMIT 5;
        """,
        conn,
    )

orders_preview

,order_id,customer_id,order_date,status,payment_type
0,3,160,2025-03-29,cancelled,card
1,150,1153,2025-11-08,cancelled,atm
2,160,2235,2025-03-30,cancelled,card
3,205,372,2025-07-12,cancelled,wallet
4,241,1493,2024-12-21,cancelled,card


## 2. 基本查詢：選欄位、排序、限制筆數

`SELECT` 決定要看哪些欄位，`ORDER BY` 決定排序，`LIMIT` 決定最多顯示幾筆。

In [10]:
with sqlite3.connect(DB_PATH) as conn:
    recent_orders = pd.read_sql_query(
        """
        SELECT order_id, customer_id, order_date, status, payment_type
        FROM orders
        ORDER BY order_date DESC
        LIMIT 10;
        """,
        conn,
    )

recent_orders

,order_id,customer_id,order_date,status,payment_type
0,2285,1511,2025-12-31,completed,card
1,2599,403,2025-12-31,completed,card
2,2875,1768,2025-12-31,completed,card
3,3308,593,2025-12-31,completed,atm
4,3688,1343,2025-12-31,cancelled,card
5,3797,1420,2025-12-31,completed,card
6,3937,860,2025-12-31,completed,atm
7,5030,346,2025-12-31,completed,card
8,5912,828,2025-12-31,completed,wallet
9,6085,1309,2025-12-31,completed,wallet


練習：把上面查詢改成最早的 10 筆訂單。

提示：把 `DESC` 改成 `ASC`。

In [13]:
with sqlite3.connect(DB_PATH) as conn:
    earliest_orders = pd.read_sql_query(
        """
        SELECT order_id, customer_id, order_date, status, payment_type
        FROM orders
        ORDER BY order_date ASC
        LIMIT 10;
        """,
        conn,
    )

earliest_orders

,order_id,customer_id,order_date,status,payment_type
0,244,2147,2024-01-01,completed,card
1,1830,2061,2024-01-01,completed,card
2,1948,1599,2024-01-01,completed,card
3,2007,2088,2024-01-01,completed,cod
4,2018,1175,2024-01-01,completed,atm
5,3255,832,2024-01-01,completed,atm
6,3572,2058,2024-01-01,completed,card
7,4002,859,2024-01-01,completed,cod
8,4045,588,2024-01-01,completed,cod
9,4704,892,2024-01-01,completed,card


## 3. 條件查詢：`WHERE`

`WHERE` 用來篩選符合條件的資料。以下查詢完成付款、付款方式為卡片、日期在 2025 年之後的訂單。

In [18]:
with sqlite3.connect(DB_PATH) as conn:
    filtered_orders = pd.read_sql_query(
        """
        SELECT order_id, customer_id, order_date, status, payment_type
        FROM orders 
        WHERE status = 'completed'
          AND payment_type = 'card'
          AND order_date >= '2025-01-01'
        ORDER BY order_date
        LIMIT 10;
        """,
        conn,
    )
#AND order_date BETWEEN '2025-01-03' AND '2025-02-01'
filtered_orders

,order_id,customer_id,order_date,status,payment_type
0,1027,1889,2025-01-01,completed,card
1,1940,1059,2025-01-01,completed,card
2,3064,1887,2025-01-01,completed,card
3,6170,598,2025-01-01,completed,card
4,10230,2246,2025-01-01,completed,card
5,14482,2357,2025-01-01,completed,card
6,14675,1448,2025-01-01,completed,card
7,15344,956,2025-01-01,completed,card
8,16339,1810,2025-01-01,completed,card
9,16943,750,2025-01-01,completed,card


文字條件也很常用。下面從 `customers` 找出城市在 Taipei、且客群為 vip 的顧客。

In [24]:
# WHERE city = 'Taipei' 
with sqlite3.connect(DB_PATH) as conn:
    taipei_vip_customers = pd.read_sql_query(
        """
        SELECT customer_id, signup_date, acquisition_channel, city, segment
        FROM customers
        WHERE city LIKE '%tai%' 
          AND segment = 'vip'
        ORDER BY signup_date DESC
        LIMIT 10;
        """,
        conn,
    )#WHERE city = 'Taipei' 

taipei_vip_customers

,customer_id,signup_date,acquisition_channel,city,segment
0,2091,2025-11-29,ads,Tainan,vip
1,1298,2025-11-17,referral,Tainan,vip
2,2071,2025-11-07,partner,Taichung,vip
3,1642,2025-11-04,organic,Tainan,vip
4,418,2025-11-02,organic,Tainan,vip
5,67,2025-10-13,referral,Tainan,vip
6,365,2025-10-13,partner,Taipei,vip
7,1877,2025-10-09,ads,Taichung,vip
8,2329,2025-09-28,referral,Taichung,vip
9,1407,2025-09-24,ads,Taipei,vip


## 4. 分組彙總：`GROUP BY`

`GROUP BY` 可以把資料依欄位分組，再搭配聚合函式計算統計值。

常見聚合函式：
- `COUNT(*)`：筆數
- `SUM(...)`：加總
- `AVG(...)`：平均
- `MIN(...)` / `MAX(...)`：最小值 / 最大值

In [25]:
with sqlite3.connect(DB_PATH) as conn:
    orders_by_payment = pd.read_sql_query(
        """
        SELECT
            payment_type,
            COUNT(*) AS order_count
        FROM orders
        WHERE status = 'completed'
        GROUP BY payment_type
        ORDER BY order_count DESC;
        """,
        conn,
    )

orders_by_payment

,payment_type,order_count
0,card,10166
1,atm,5111
2,wallet,3076
3,cod,2060


接著把 `orders`、`order_items`、`products` 串起來，計算不同商品類別的營收。這段多了 `JOIN`，但核心仍然是 `GROUP BY category`。

In [ ]:
with sqlite3.connect(DB_PATH) as conn:
    revenue_by_category = pd.read_sql_query(
        """
        SELECT
            p.category,
            COUNT(DISTINCT o.order_id) AS order_count,
            SUM(oi.quantity * oi.unit_price * (1 - oi.discount_rate)) AS revenue,
            AVG(oi.quantity * oi.unit_price * (1 - oi.discount_rate)) AS avg_item_revenue
        FROM orders AS o
        JOIN order_items AS oi
          ON o.order_id = oi.order_id
        JOIN products AS p
          ON oi.product_id = p.product_id
        WHERE o.status = 'completed'
        GROUP BY p.category
        ORDER BY revenue DESC;
        """,# DISTINCT--> 去重覆
        conn,
    )

revenue_by_category

,category,order_count,revenue,avg_item_revenue
0,beauty,5493,18283494.35,3027.569854
1,home,5569,18061948.35,2926.907851
2,sports,5466,17428979.90,2874.646198
3,electronics,5583,17368081.05,2798.595077
4,grocery,5584,16824601.25,2722.427387
5,fashion,5520,13804694.85,2254.932187


練習：請改寫上一段，改成依 `payment_type` 統計營收。

答案範例在下一格。

In [27]:
with sqlite3.connect(DB_PATH) as conn:
    revenue_by_payment = pd.read_sql_query(
        """
        SELECT
            o.payment_type,
            COUNT(DISTINCT o.order_id) AS order_count,
            SUM(oi.quantity * oi.unit_price * (1 - oi.discount_rate)) AS revenue
        FROM orders AS o
        JOIN order_items AS oi
          ON o.order_id = oi.order_id
        WHERE o.status = 'completed'
        GROUP BY o.payment_type
        ORDER BY revenue DESC;
        """,
        conn,
    )

revenue_by_payment

,payment_type,order_count,revenue
0,card,10166,50567692.45
1,atm,5111,25452594.55
2,wallet,3076,15343032.20
3,cod,2060,10408480.55


## 5. 新增資料：`CREATE TABLE` 與 `INSERT`

為了避免修改原始資料庫，這裡建立一個記憶體資料庫。只要 notebook 關掉，這張練習表就會消失。

In [29]:
practice_conn = sqlite3.connect(":memory:")

practice_conn.execute(
    """
    CREATE TABLE students (
        student_id INTEGER PRIMARY KEY,
        name TEXT NOT NULL,
        city TEXT,
        score INTEGER
    );
    """
)


In [34]:
practice_conn.execute(
    """
    INSERT INTO students (student_id, name, city, score)
    VALUES (2, 'Amy2', 'Taipei', 88),(3, 'Amy3', 'Taipei', 88);

    """
)
#    INSERT INTO students (student_id, name)
#    VALUES (2, 'Ben');

In [37]:

practice_conn.executemany(
    """
    INSERT INTO students (student_id, name, city, score)
    VALUES (?, ?, ?, ?);
    """,
    [
        (112, 'Ben', 'Tainan', 75),
        (113, 'Cindy', 'Taipei', 92),
        (114, 'David', 'Hsinchu', 81),
    ],
)
practice_conn.commit()#開始執行
pd.read_sql_query("""
                  SELECT * 
                  FROM students 
                  ORDER BY student_id;
                  """, 
                  practice_conn)

,student_id,name,city,score
0,1,Ben,NaN,NaN
1,2,Amy2,Taipei,88.0
2,3,Amy3,Taipei,88.0
3,12,Ben,Tainan,75.0
4,13,Cindy,Taipei,92.0
5,14,David,Hsinchu,81.0
6,112,Ben,Tainan,75.0
7,113,Cindy,Taipei,92.0
8,114,David,Hsinchu,81.0


新增後也可以馬上做條件查詢與分組統計。

In [38]:
pd.read_sql_query(
    """
    SELECT
        city,
        COUNT(*) AS student_count,
        AVG(score) AS avg_score
    FROM students
    GROUP BY city
    ORDER BY avg_score DESC;
    """,
    practice_conn,
)

,city,student_count,avg_score
0,Taipei,4,90.0
1,Hsinchu,2,81.0
2,Tainan,2,75.0
3,NaN,1,NaN


最後把練習用連線關掉。

In [39]:
practice_conn.close()

## 常見錯誤與延伸

常見錯誤：SQL 字串裡的文字值要用引號，例如 `status = 'completed'`。如果寫成 `status = completed`，SQLite 會把 `completed` 當成欄位名稱。

延伸練習：
- 用 `sessions` 表統計不同 `device` 的 session 數量
- 用 `customers` 表統計不同 `city` 的顧客數量
- 把 `LIMIT 10` 改成 `LIMIT 20`，觀察結果是否更容易判讀